In [ ]:
# ==========================================
# Import Libraries
# ==========================================

import pandas as pd
import numpy as np

from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler

import joblib
import os

print("Libraries Imported Successfully")

In [ ]:
# ==========================================
# Project Paths
# ==========================================

DATA_PATH = "../data/processed/"
MODEL_PATH = "../models/"

os.makedirs(MODEL_PATH, exist_ok=True)

In [ ]:
# ==========================================
# Load Dataset
# ==========================================

df = pd.read_csv(
    DATA_PATH + "eda_dataset.csv"
)

print(df.shape)

df.head()

In [ ]:
date_columns = [
    col for col in df.columns
    if "date" in col.lower()
]

date_column = date_columns[0]

df[date_column] = pd.to_datetime(df[date_column])

In [ ]:
df["Year"] = df[date_column].dt.year

In [ ]:
df["Month"] = df[date_column].dt.month

In [ ]:
df["Quarter"] = df[date_column].dt.quarter

In [ ]:
df["Week"] = df[date_column].dt.isocalendar().week.astype(int)

In [ ]:
df["Day"] = df[date_column].dt.day

In [ ]:
df["DayOfWeek"] = df[date_column].dt.dayofweek

In [ ]:
df["Weekend"] = df["DayOfWeek"].isin([5,6]).astype(int)

In [ ]:
if "price" in df.columns and "quantity" in df.columns:

    df["Revenue"] = df["price"] * df["quantity"]

In [ ]:
if "cost" in df.columns:

    df["Profit"] = df["Revenue"] - (df["cost"] * df["quantity"])

In [ ]:
if "Profit" in df.columns:

    df["Profit_Margin"] = (
        df["Profit"] /
        df["Revenue"]
    ) * 100

In [ ]:
if "stock_quantity" in df.columns:

    df["Inventory_Ratio"] = (

        df["quantity"] /

        (df["stock_quantity"] + 1)

    )

In [ ]:
promotion_columns = [

col for col in df.columns

if "promotion" in col.lower()

]

if promotion_columns:

    df["Promotion_Flag"] = (

        df[promotion_columns[0]]

        .notnull()

        .astype(int)

    )

In [ ]:
if "quantity" in df.columns:

    df = df.sort_values(date_column)

    df["Lag_1"] = df["quantity"].shift(1)

    df["Lag_7"] = df["quantity"].shift(7)

In [ ]:
df["Rolling_Mean_7"] = (

    df["quantity"]

    .rolling(7)

    .mean()

)

In [ ]:
df["Rolling_STD_7"] = (

    df["quantity"]

    .rolling(7)

    .std()

)

In [ ]:
if "customer_id" in df.columns:

    purchase_count = (

        df.groupby("customer_id")

        .size()

        .reset_index(name="Purchase_Count")

    )

    df = df.merge(

        purchase_count,

        on="customer_id",

        how="left"

    )

In [ ]:
if "customer_id" in df.columns:

    lifetime = (

        df.groupby("customer_id")

        ["Revenue"]

        .sum()

        .reset_index(name="Customer_Lifetime_Value")

    )

    df = df.merge(

        lifetime,

        on="customer_id",

        how="left"

    )

In [ ]:
if "sku_id" in df.columns:

    freq = (

        df.groupby("sku_id")

        .size()

        .reset_index(name="Sales_Frequency")

    )

    df = df.merge(

        freq,

        on="sku_id",

        how="left"

    )

In [ ]:
df.fillna(0, inplace=True)

In [ ]:
encoder = LabelEncoder()

categorical = df.select_dtypes(include="object").columns

for col in categorical:

    df[col] = encoder.fit_transform(df[col].astype(str))

In [ ]:
numeric_columns = df.select_dtypes(include=np.number).columns

numeric_columns

In [ ]:
scaler = StandardScaler()

df[numeric_columns] = scaler.fit_transform(

    df[numeric_columns]

)

In [ ]:
joblib.dump(

    scaler,

    MODEL_PATH + "scaler.pkl"

)

print("Scaler Saved")

In [ ]:
variance = (

    df[numeric_columns]

    .var()

    .sort_values(ascending=False)

)

variance.head(20)

In [ ]:
print(df.shape)

df.head()

In [ ]:
df.to_csv(

    DATA_PATH +

    "feature_engineered.csv",

    index=False

)

print("feature_engineered.csv Saved")

In [ ]:
print("="*60)

print("PHASE 4 COMPLETED")

print("="*60)

print("""
✔ Time Features Created
✔ Revenue Features Created
✔ Profit Features Created
✔ Inventory Features Created
✔ Customer Features Created
✔ Promotion Features Created
✔ Lag Features Created
✔ Rolling Features Created
✔ Label Encoding Completed
✔ Feature Scaling Completed
✔ Feature Engineered Dataset Saved
✔ Scaler Saved
""")